In [1]:
import os
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# 1. LOAD DATASETS DYNAMICALLY
train_path, test_path = None, None
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == 'train.csv' and not train_path:
            train_path = os.path.join(dirname, filename)
        elif filename == 'test.csv' and not test_path:
            test_path = os.path.join(dirname, filename)

train_path = train_path or '/kaggle/input/train.csv'
test_path = test_path or '/kaggle/input/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# 2. ADVANCED FEATURE ENGINEERING PIPELINE (V2)
def advanced_feature_engineering(df, is_train=True, train_route_freq=None):
    df = df.copy()
    
    if is_train:
        if 'total_amount' in df.columns:
            df = df[df['total_amount'] > 0].reset_index(drop=True)
            fare_cap = df['total_amount'].quantile(0.999)
            df = df[df['total_amount'] <= fare_cap].reset_index(drop=True)
        if 'trip_distance' in df.columns:
            df = df[df['trip_distance'] > 0].reset_index(drop=True)

    pickup_col = next((col for col in ['pickup_datetime', 'tpep_pickup_datetime'] if col in df.columns), None)
    dropoff_col = next((col for col in ['dropoff_datetime', 'tpep_dropoff_datetime'] if col in df.columns), None)
    
    if pickup_col:
        df[pickup_col] = pd.to_datetime(df[pickup_col])
        df['hour'] = df[pickup_col].dt.hour
        df['dayofweek'] = df[pickup_col].dt.dayofweek
        df['month'] = df[pickup_col].dt.month
        df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
        df['is_rush_hour'] = ((~df['is_weekend'].astype(bool)) & ((df['hour'].between(8, 10)) | (df['hour'].between(17, 20)))).astype(int)

    if pickup_col and dropoff_col:
        df[dropoff_col] = pd.to_datetime(df[dropoff_col])
        df['trip_duration_min'] = (df[dropoff_col] - df[pickup_col]).dt.total_seconds() / 60.0
        df['speed_mph'] = df['trip_distance'] / ((df['trip_duration_min'] / 60.0) + 1e-5)
        df['distance_per_min'] = df['trip_distance'] / (df['trip_duration_min'] + 1e-5)

    surcharge_cols = ['extra', 'tolls_amount', 'improvement_surcharge', 'congestion_surcharge', 'Airport_fee']
    existing_surcharges = [col for col in surcharge_cols if col in df.columns]
    if existing_surcharges:
        df['total_surcharges'] = df[existing_surcharges].sum(axis=1)

    if 'PULocationID' in df.columns and 'DOLocationID' in df.columns:
        df['route_pair'] = df['PULocationID'].astype(str) + "_" + df['DOLocationID'].astype(str)
        
        if is_train:
            route_freq = df['route_pair'].value_counts().to_dict()
            df['route_freq'] = df['route_pair'].map(route_freq).fillna(1)
        else:
            route_freq = train_route_freq if train_route_freq is not None else {}
            df['route_freq'] = df['route_pair'].map(route_freq).fillna(1)
            
        df = df.drop(columns=['route_pair'], errors='ignore')

    return df, (route_freq if is_train else None)

train_clean, train_route_freq = advanced_feature_engineering(train_df, is_train=True)
test_clean, _ = advanced_feature_engineering(test_df, is_train=False, train_route_freq=train_route_freq)

# 3. TRAINING & VALIDATION WITH CATBOOST
target = 'total_amount'
exclude_cols = [target, 'id', 'pickup_datetime', 'dropoff_datetime', 'tpep_pickup_datetime', 'tpep_dropoff_datetime']

train_numeric = train_clean.select_dtypes(include=[np.number]).columns.tolist()
test_numeric = test_clean.select_dtypes(include=[np.number]).columns.tolist()
features = [col for col in train_numeric if col in test_numeric and col not in exclude_cols]

print(f"\nTraining CatBoost using {len(features)} features: {features}")

X = train_clean[features].astype(np.float32)
y = train_clean[target].astype(np.float32)
X_test = test_clean[features].astype(np.float32)

SEED = 42
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

oof_preds_cb = np.zeros(len(X))
test_preds_cb = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    cb_model = CatBoostRegressor(
        iterations=1500,
        learning_rate=0.03,
        depth=6,
        random_seed=SEED,
        verbose=0
    )
    
    cb_model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=50,
        verbose=False
    )
    
    oof_preds_cb[val_idx] = cb_model.predict(X_val)
    test_preds_cb += cb_model.predict(X_test) / kf.n_splits

cv_rmse_cb = np.sqrt(mean_squared_error(y, oof_preds_cb))
print(f"\nCatBoost Out-of-Fold RMSE: {cv_rmse_cb:.4f}")

# 4. GENERATE SUBMISSION
id_col = 'id' if 'id' in test_clean.columns else test_clean.columns[0]
submission = pd.DataFrame({
    id_col: test_clean[id_col],
    'total_amount': test_preds_cb
})

submission.to_csv('submission.csv', index=False)
print("\nCatBoost submission file 'submission.csv' generated successfully!")
display(submission.head())


Training CatBoost using 22 features: ['VendorID', 'passenger_count', 'trip_distance', 'RatecodeID', 'PULocationID', 'DOLocationID', 'extra', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'congestion_surcharge', 'Airport_fee', 'hour', 'dayofweek', 'month', 'is_weekend', 'is_rush_hour', 'trip_duration_min', 'speed_mph', 'distance_per_min', 'total_surcharges', 'route_freq']

CatBoost Out-of-Fold RMSE: 4.7983

CatBoost submission file 'submission.csv' generated successfully!


,id,total_amount
0,0,31.496319
1,1,11.546661
2,2,21.154065
3,3,21.035720
4,4,20.307834
